In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
import pandas as pd

DATA_PATH = "/kaggle/input/competitions/ieee-fraud-detection"

train = pd.read_csv(
    f"{DATA_PATH}/train_transaction.csv"
)

print("Shape:", train.shape)
print("Fraud cases:", train["isFraud"].sum())
print("Fraud rate:", train["isFraud"].mean())

Shape: (590540, 394)
Fraud cases: 20663
Fraud rate: 0.03499000914417313


In [4]:
print(train.info())
print(train.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB
None
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  .

In [5]:
# Basic inspection
print("Shape:", train.shape)
print("\nFraud distribution:")
print(train["isFraud"].value_counts())

print("\nFraud percentage:")
print(train["isFraud"].mean() * 100)

print("\nData types:")
print(train.dtypes.value_counts())

print("\nMissing values — top 20:")
print(train.isna().sum().sort_values(ascending=False).head(20))

Shape: (590540, 394)

Fraud distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud percentage:
3.4990009144173126

Data types:
float64    376
object      14
int64        4
Name: count, dtype: int64

Missing values — top 20:
dist2    552913
D7       551623
D13      528588
D14      528353
D12      525823
D6       517353
D9       515614
D8       515614
V153     508595
V149     508595
V141     508595
V146     508595
V154     508595
V162     508595
V142     508595
V158     508595
V161     508595
V157     508595
V138     508595
V139     508595
dtype: int64


In [6]:
# Memory optimization
float_cols = train.select_dtypes(include=["float64"]).columns
int_cols = train.select_dtypes(include=["int64"]).columns

train[float_cols] = train[float_cols].astype("float32")
train[int_cols] = train[int_cols].astype("int32")

print("Memory optimization complete.")

Memory optimization complete.


In [7]:
train = train.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(train) * 0.80)

train_df = train.iloc[:split_idx].copy()
valid_df = train.iloc[split_idx:].copy()

print("Training:", train_df.shape)
print("Validation:", valid_df.shape)

print("\nLatest training time:")
print(train_df["TransactionDT"].max())

print("\nEarliest validation time:")
print(valid_df["TransactionDT"].min())

Training: (472432, 394)
Validation: (118108, 394)

Latest training time:
12192842

Earliest validation time:
12192900


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

X_train = train_df[["TransactionDT"]]
X_valid = valid_df[["TransactionDT"]]

y_train = train_df["isFraud"]
y_valid = valid_df["isFraud"]

dummy = DummyClassifier(strategy="prior")

dummy.fit(X_train, y_train)

dummy_probs = dummy.predict_proba(X_valid)[:, 1]

dummy_pr_auc = average_precision_score(y_valid, dummy_probs)

print("Dummy PR-AUC:", dummy_pr_auc)

Dummy PR-AUC: 0.034409184813899145


In [9]:
import numpy as np

features = [
    "TransactionAmt",
    "ProductCD",
    "card4",
    "card6"
]

X_train = train_df[features].copy()
X_valid = valid_df[features].copy()

y_train = train_df["isFraud"]
y_valid = valid_df["isFraud"]

X_train["TransactionAmt"] = np.log1p(X_train["TransactionAmt"])
X_valid["TransactionAmt"] = np.log1p(X_valid["TransactionAmt"])

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

numeric_features = ["TransactionAmt"]

categorical_features = [
    "ProductCD",
    "card4",
    "card6"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

logistic_probs = logistic_model.predict_proba(X_valid)[:, 1]

logistic_pr_auc = average_precision_score(
    y_valid,
    logistic_probs
)

logistic_roc_auc = roc_auc_score(
    y_valid,
    logistic_probs
)

print("Logistic PR-AUC:", logistic_pr_auc)
print("Logistic ROC-AUC:", logistic_roc_auc)

Logistic PR-AUC: 0.12321600517593911
Logistic ROC-AUC: 0.7433701262853514


In [11]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

tree_probs = tree_model.predict_proba(X_valid)[:, 1]

tree_pr_auc = average_precision_score(
    y_valid,
    tree_probs
)

tree_roc_auc = roc_auc_score(
    y_valid,
    tree_probs
)

print("Decision Tree PR-AUC:", tree_pr_auc)
print("Decision Tree ROC-AUC:", tree_roc_auc)

Decision Tree PR-AUC: 0.11689999464059513
Decision Tree ROC-AUC: 0.7402850244618253


In [12]:
!pip install lightgbm -q

In [13]:
from lightgbm import LGBMClassifier

X_train_lgb = train_df[features].copy()
X_valid_lgb = valid_df[features].copy()

for col in ["ProductCD", "card4", "card6"]:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_valid_lgb[col] = X_valid_lgb[col].astype("category")

y_train = train_df["isFraud"]
y_valid = valid_df["isFraud"]

In [14]:
boost_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

boost_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=[
        "ProductCD",
        "card4",
        "card6"
    ]
)

boost_probs = boost_model.predict_proba(
    X_valid_lgb
)[:, 1]

boost_pr_auc = average_precision_score(
    y_valid,
    boost_probs
)

boost_roc_auc = roc_auc_score(
    y_valid,
    boost_probs
)

print("LightGBM PR-AUC:", boost_pr_auc)
print("LightGBM ROC-AUC:", boost_roc_auc)

LightGBM PR-AUC: 0.13827094708929535
LightGBM ROC-AUC: 0.7668264191079587


In [15]:
features_52 = [
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",

    "card1", "card2", "card3", "card4",
    "card5", "card6",

    "addr1", "addr2",

    "P_emaildomain",
    "R_emaildomain",

    "C1", "C2", "C3", "C4", "C5", "C6", "C7",
    "C8", "C9", "C10", "C11", "C12", "C13", "C14",

    "D1", "D2", "D3", "D4", "D5", "D6", "D7",
    "D8", "D9", "D10", "D11", "D12", "D13", "D14",
    "D15",

    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9",

    "dist1"
]

data_52 = train[features_52 + ["isFraud"]].copy()

print(data_52.shape)

(590540, 53)


In [16]:
data_52 = data_52.sort_values(
    "TransactionDT"
).reset_index(drop=True)

split_idx = int(len(data_52) * 0.80)

train_52 = data_52.iloc[:split_idx]
valid_52 = data_52.iloc[split_idx:]

X_train_52 = train_52.drop(columns=["isFraud"])
X_valid_52 = valid_52.drop(columns=["isFraud"])

y_train_52 = train_52["isFraud"]
y_valid_52 = valid_52["isFraud"]

categorical_cols = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
    "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9"
]

for col in categorical_cols:
    X_train_52[col] = X_train_52[col].astype("category")
    X_valid_52[col] = X_valid_52[col].astype("category")

print("Train:", X_train_52.shape)
print("Validation:", X_valid_52.shape)

Train: (472432, 52)
Validation: (118108, 52)


In [17]:
boost_model_52 = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

boost_model_52.fit(
    X_train_52,
    y_train_52,
    categorical_feature=categorical_cols
)

probs_52 = boost_model_52.predict_proba(
    X_valid_52
)[:, 1]

pr_auc_52 = average_precision_score(
    y_valid_52,
    probs_52
)

roc_auc_52 = roc_auc_score(
    y_valid_52,
    probs_52
)

print("LightGBM 52-feature PR-AUC:", pr_auc_52)
print("LightGBM 52-feature ROC-AUC:", roc_auc_52)

LightGBM 52-feature PR-AUC: 0.526653856626605
LightGBM 52-feature ROC-AUC: 0.9090549409700828


In [18]:
importance = pd.DataFrame({
    "feature": X_train_52.columns,
    "importance": boost_model_52.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(
    importance.head(20).to_string(index=False)
)

       feature  importance
         card2         597
         card1         590
 TransactionDT         509
         addr1         476
TransactionAmt         459
 P_emaildomain         432
           D15         352
           C13         320
            D2         311
            C1         305
         card5         259
            D1         243
           D10         240
            D8         215
            D4         213
 R_emaildomain         206
           C14         200
            C2         196
            C6         194
         dist1         181


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

numeric_cols_52 = [
    col for col in X_train_52.columns
    if col not in categorical_cols
]

numeric_pipeline_52 = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline_52 = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor_52 = ColumnTransformer([
    ("num", numeric_pipeline_52, numeric_cols_52),
    ("cat", categorical_pipeline_52, categorical_cols)
])

In [20]:
logistic_52 = Pipeline([
    ("preprocessor", preprocessor_52),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

logistic_52.fit(
    X_train_52,
    y_train_52
)

logistic_probs_52 = logistic_52.predict_proba(
    X_valid_52
)[:, 1]

logistic_pr_auc_52 = average_precision_score(
    y_valid_52,
    logistic_probs_52
)

logistic_roc_auc_52 = roc_auc_score(
    y_valid_52,
    logistic_probs_52
)

print("Logistic 52-feature PR-AUC:",
      logistic_pr_auc_52)

print("Logistic 52-feature ROC-AUC:",
      logistic_roc_auc_52)

Logistic 52-feature PR-AUC: 0.2777727172464165
Logistic 52-feature ROC-AUC: 0.8147503034986912


In [21]:
from sklearn.tree import DecisionTreeClassifier

tree_52 = Pipeline([
    ("preprocessor", preprocessor_52),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ))
])

tree_52.fit(
    X_train_52,
    y_train_52
)

tree_probs_52 = tree_52.predict_proba(
    X_valid_52
)[:, 1]

tree_pr_auc_52 = average_precision_score(
    y_valid_52,
    tree_probs_52
)

tree_roc_auc_52 = roc_auc_score(
    y_valid_52,
    tree_probs_52
)

print("Decision Tree 52-feature PR-AUC:",
      tree_pr_auc_52)

print("Decision Tree 52-feature ROC-AUC:",
      tree_roc_auc_52)

Decision Tree 52-feature PR-AUC: 0.3193118104633794
Decision Tree 52-feature ROC-AUC: 0.788441084358724


In [22]:
results = pd.DataFrame({
    "Model": [
        "Dummy",
        "Logistic Regression",
        "Decision Tree",
        "LightGBM"
    ],
    "Features": [
        "Baseline",
        "52",
        "52",
        "52"
    ],
    "PR-AUC": [
        dummy_pr_auc,
        logistic_pr_auc_52,
        tree_pr_auc_52,
        pr_auc_52
    ],
    "ROC-AUC": [
        dummy_pr_auc,
        logistic_roc_auc_52,
        tree_roc_auc_52,
        roc_auc_52
    ]
})

print(results.to_string(index=False))

              Model Features   PR-AUC  ROC-AUC
              Dummy Baseline 0.034409 0.034409
Logistic Regression       52 0.277773 0.814750
      Decision Tree       52 0.319312 0.788441
           LightGBM       52 0.526654 0.909055
